In [ ]:
import re
import math
from collections import defaultdict, Counter


pages = [
    "Invoice Number: 101\nCustomer: Rahul\nPage 1\nTotal Amount: 5000",
    "Invoice Number: 101\nCustomer: Rahul\nPage 2\nPayment Details",
    "Report Title: Annual Report\nPage 1\nCompany Performance",
    "Report Title: Annual Report\nPage 2\nFinancial Results",
    "Invoice Number: 102\nCustomer: Amit\nPage 1\nTotal Amount: 3000"
]


def extract_features(text):
    return {
        "invoice": "invoice" in text.lower(),
        "report": "report" in text.lower(),
        "page1": bool(re.search(r"page\s*1\b", text.lower())),
        "page2": bool(re.search(r"page\s*2\b", text.lower())),
        "new_number": bool(re.search(r"number:\s*\d+", text.lower())),
        "length": "long" if len(text) > 60 else "short"
    }


training_data = [
    [
        ({"invoice": True, "report": False, "page1": True,
          "page2": False, "new_number": True, "length": "long"}, "START"),
        ({"invoice": True, "report": False, "page1": False,
          "page2": True, "new_number": True, "length": "short"}, "CONTINUE")
    ],
    [
        ({"invoice": False, "report": True, "page1": True,
          "page2": False, "new_number": False, "length": "short"}, "START"),
        ({"invoice": False, "report": True, "page1": False,
          "page2": True, "new_number": False, "length": "short"}, "CONTINUE")
    ],
    [
        ({"invoice": True, "report": False, "page1": True,
          "page2": False, "new_number": True, "length": "short"}, "START"),
        ({"invoice": False, "report": True, "page1": True,
          "page2": False, "new_number": False, "length": "short"}, "START")
    ]
]

states = ["START", "CONTINUE"]

def train_hmm(data):
    transition = defaultdict(Counter)
    emission = defaultdict(lambda: defaultdict(Counter))
    state_count = Counter()

    for sequence in data:
        previous = None

        for features, state in sequence:
            state_count[state] += 1

            if previous:
                transition[previous][state] += 1

            for key, value in features.items():
                emission[state][key][value] += 1

            previous = state

    return transition, emission, state_count

transition, emission, state_count = train_hmm(training_data)


def emission_probability(features, state):
    probability = 1.0

    for key, value in features.items():
        counts = emission[state][key]
        total = sum(counts.values())

        probability *= (
            counts[value] + 1
        ) / (total + len(counts) + 1)

    return probability


def viterbi(sequence):
    dp = [{}]
    path = {}

    for state in states:
        dp[0][state] = (
            math.log(state_count[state] + 1)
            + math.log(emission_probability(sequence[0], state))
        )
        path[state] = [state]

    for t in range(1, len(sequence)):
        dp.append({})
        new_path = {}

        for current in states:
            candidates = []

            for previous in states:
                trans = (
                    transition[previous][current] + 1
                ) / (
                    sum(transition[previous].values()) + len(states)
                )

                score = (
                    dp[t-1][previous]
                    + math.log(trans)
                    + math.log(emission_probability(
                        sequence[t], current
                    ))
                )

                candidates.append((score, previous))

            best_score, best_previous = max(candidates)

            dp[t][current] = best_score
            new_path[current] = path[best_previous] + [current]

        path = new_path

    best_state = max(dp[-1], key=dp[-1].get)

    return path[best_state]


features = [extract_features(page) for page in pages]


predictions = viterbi(features)


print("DOCUMENT SEPARATION RESULTS")
print("--------------------------------")

document_number = 0

for i, (page, label) in enumerate(zip(pages, predictions)):

    if label == "START":
        document_number += 1

    print(f"Page {i+1}: {label}")
    print(f"Document Number: {document_number}")
    print("--------------------------------")

print("Total Documents:", document_number)

DOCUMENT SEPARATION RESULTS
--------------------------------
Page 1: START
Document Number: 1
--------------------------------
Page 2: CONTINUE
Document Number: 1
--------------------------------
Page 3: START
Document Number: 2
--------------------------------
Page 4: CONTINUE
Document Number: 2
--------------------------------
Page 5: START
Document Number: 3
--------------------------------
Total Documents: 3
